# Pipeline de Entrenamiento _(version detallada)_

_Cerrar el ciclo: del feature store al modelo de regresion entrenado y evaluado, con consistencia entrenamiento-serving._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Pipeline de Entrenamiento](assets/header.png)

> 📖 **Version detallada** — Incluye comentarios linea a linea en cada bloque de codigo
> y ampliaciones teoricas en las secciones de contexto.  
> Para una lectura mas rapida, consulta la version original `04_pipeline_entrenamiento.ipynb`.

## Introduccion

En el Notebook 3 definimos features en **Feast** y aprendimos a materializarlas y
servirlas. Pero un feature store no es un fin en si mismo: existe para
**alimentar modelos**. En este notebook **cerramos el ciclo**:

```
   feature store  ->  set de entrenamiento  ->  modelo de regresion entrenado y evaluado
```

Es decir:

1. Construimos el **set de entrenamiento** leyendo del **offline store** de Feast
   con `get_historical_features` (con un fallback a parquet si Feast no esta
   disponible).
2. Entrenamos un **regresor** que predice `SalePrice` (entrenado sobre
   `log1p(SalePrice)`, como en la competencia de Kaggle).
3. Lo **evaluamos** con **RMSE, MAE y R2**, ademas del **RMSE sobre `log(SalePrice)`**
   (la metrica oficial de Kaggle), y graficamos **predicho vs real**.

> **La idea grande: consistencia entrenamiento-serving.** El set de entrenamiento
> sale de las *mismas* definiciones de features que luego serviran online. Por eso
> el modelo ve en produccion exactamente la misma representacion con la que
> aprendio: no hay sorpresas de *skew*. El feature store es el contrato que une
> ambos mundos.

### ¿Por que log1p(SalePrice)?

`SalePrice` tiene una distribucion muy sesgada a la derecha (skewness ≈ 1.88).
Si entrenamos directamente sobre los precios en dolares, el modelo optimiza el
**error absoluto en dolares** — lo que hace que una casa de $500k errando en
$50k pese mucho mas que una de $100k errando en $20k.

Al entrenar sobre `log1p(SalePrice)` optimizamos el **error relativo** (porcentual),
que es la metrica de Kaggle (RMSLE). Para predecir el precio en dolares al final,
aplicamos la inversa: `np.expm1(prediccion_log)`.

## Setup

Definimos las rutas y un helper para localizar el repo de Feast y el parquet del
offline store. No se necesita ningun servicio levantado para la ruta de parquet;
para usar Feast directamente, primero corre `feast apply` (Notebook 3).

In [ ]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

# REPO: directorio raiz del feature repo de Feast.
# Usamos os.path.abspath + join para que la ruta sea absoluta
# independientemente de desde donde se ejecute el notebook.
REPO = os.path.abspath(os.path.join("..", "platform", "feature_repo"))

# PARQUET_PATH: ruta al archivo parquet del OFFLINE store.
# Este archivo fue creado en el Notebook 3 con feat.to_parquet().
# Contiene todas las features historicas con sus event_timestamps.
PARQUET_PATH = os.path.join(REPO, "data", "housing_features.parquet")

print("repo de features:", REPO)
print("parquet offline :", PARQUET_PATH)
print("existe parquet  :", os.path.exists(PARQUET_PATH))

## Paso 1 — construir el set de entrenamiento desde el feature store

La forma *correcta* de armar un set de entrenamiento es con
`get_historical_features`: le pasas un **dataframe de entidades** (que casas y
en que `event_timestamp`) y te devuelve las features validas en ese momento
(point-in-time). Aqui usamos como timestamp "ahora", asi que traemos el ultimo
valor de cada casa.

Si Feast no esta instalado o el registry no esta aplicado, caemos a **leer el
parquet del offline store directamente** (la misma fuente que Feast usa por
debajo). En ambos casos obtenemos las mismas features.

### ¿Que es point-in-time correctness?

Imagina que tienes historico de features desde enero a diciembre y quieres entrenar
un modelo para predecir el precio de cada casa en el mes de su venta. Si usas
las features del futuro (p.ej. mejoras que se hicieron despues), introduces
**label leakage**. `get_historical_features` garantiza que para cada fila en el
`entity_df`, solo usa features cuyo `event_timestamp` <= el timestamp de la etiqueta.

In [ ]:
def load_offline_parquet():
    """Fallback: lee directamente el parquet del offline store de Feast."""
    # Leemos el parquet que Feast usa como fuente de datos historicos.
    # Contiene las mismas features que devuelve get_historical_features.
    df = pd.read_parquet(PARQUET_PATH)
    print(f"[parquet] {len(df)} filas leidas de {PARQUET_PATH}")
    return df

# Lista de features con formato 'feature_view:nombre_feature'.
# Estos son los mismos nombres definidos en platform/feature_repo/features.py.
FEATURES = [
    "house_features:overall_qual",
    "house_features:gr_liv_area",
    "house_features:total_bsmt_sf",
    "house_features:first_flr_sf",
    "house_features:garage_cars",
    "house_features:garage_area",
    "house_features:year_built",
    "house_features:lot_area",
    "house_features:full_bath",
    "house_features:neighborhood",
    "house_features:exter_qual_ord",
]

def build_training_set():
    """Intenta Feast (get_historical_features); si falla, usa el parquet."""
    try:
        from feast import FeatureStore
        # Instanciamos el FeatureStore apuntando al directorio del repo de features.
        # Esto lee el registry (registry.db o Postgres) para conocer las definiciones.
        store = FeatureStore(repo_path=REPO)

        # Cargamos el parquet para obtener las entidades (house_id) y el objetivo.
        base = pd.read_parquet(PARQUET_PATH)

        # entity_df: le decimos a Feast QUE casas y EN QUE MOMENTO queremos features.
        # event_timestamp=utcnow() -> "trae el ultimo valor disponible para cada casa".
        entity_df = pd.DataFrame({
            "house_id": base["house_id"].values,
            "event_timestamp": [datetime.utcnow()] * len(base),
        })

        # get_historical_features: hace un join point-in-time-correcto.
        # Para cada fila de entity_df, busca el valor de feature con
        # event_timestamp <= el timestamp de la entidad y dentro del TTL.
        feats = store.get_historical_features(
            entity_df=entity_df, features=FEATURES,
        ).to_df()

        # El objetivo (sale_price) no es una feature que se sirve online;
        # lo adjuntamos desde el parquet original.
        feats = feats.merge(base[["house_id", "sale_price"]], on="house_id")
        print(f"[feast] set de entrenamiento point-in-time: {feats.shape}")
        return feats
    except Exception as e:
        # Si Feast no esta instalado o no se ha ejecutado 'feast apply',
        # usamos el parquet directamente (misma fuente de datos).
        print(f"[feast] no disponible ({type(e).__name__}: {e}); uso el parquet.")
        return load_offline_parquet()

train_df = build_training_set()
train_df.head()

## Paso 2 — entrenar un regresor

Predecimos `SalePrice` con un **`RandomForestRegressor`** dentro de un `Pipeline`
de sklearn. Entrenamos sobre **`log1p(SalePrice)`** (asi el modelo optimiza el error
en escala logaritmica, que es la metrica de Kaggle y reduce el peso de las casas
caras). El pipeline encapsula el preprocesamiento (one-hot del barrio) junto con el
modelo, de modo que *exactamente los mismos pasos* corren en entrenamiento y en
serving. Eso es, de nuevo, consistencia entrenamiento-serving, ahora a nivel de
codigo del modelo.

### ¿Por que RandomForest dentro de un Pipeline?

Un `Pipeline` de sklearn encadena transformadores y un modelo final. Cuando hacemos
`model.fit(X_train, y_train)`, el pipeline llama `fit_transform` en cada
transformador y `fit` en el modelo. Cuando llamamos `model.predict(X_test)`, solo
llama `transform` en los transformadores. Esto garantiza que los parametros de
preprocesamiento (categorias del OneHotEncoder) se aprendan solo del train.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Definimos las features numericas (ya son numeros en el parquet)
# y las categoricas (requieren one-hot encoding).
numeric = ["overall_qual", "gr_liv_area", "total_bsmt_sf", "first_flr_sf",
           "garage_cars", "garage_area", "year_built", "lot_area",
           "full_bath", "exter_qual_ord"]
categorical = ["neighborhood"]

# Filtramos solo las columnas que existen en el DataFrame
# (robusto al fallback de parquet que puede tener columnas extra).
numeric = [c for c in numeric if c in train_df.columns]
categorical = [c for c in categorical if c in train_df.columns]

# X: matriz de features (numericas + categoricas).
X = train_df[numeric + categorical].copy()

# y: precio en dolares (objetivo real).
y = train_df["sale_price"].astype(float)

# y_log: log1p del precio — es lo que el modelo OPTIMIZA directamente.
# Entrenar en escala log hace que el modelo trate igual un error del 10%
# en una casa barata que en una cara (metrica de Kaggle).
y_log = np.log1p(y)

# Separamos en train (75%) y test (25%) con semilla fija para reproducibilidad.
# Separamos ANTES de cualquier fit para evitar data leakage.
X_train, X_test, y_train, y_test, ylog_train, ylog_test = train_test_split(
    X, y, y_log, test_size=0.25, random_state=42,
)

# ColumnTransformer: aplica OneHotEncoder solo a las columnas categoricas;
# las numericas pasan tal cual (remainder="passthrough").
pre = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough",  # las columnas numericas no se transforman
)

# Pipeline: (1) preprocesamiento -> (2) modelo.
# Durante fit: OneHotEncoder aprende las categorias de 'neighborhood' en train.
# Durante predict: aplica el mismo encoding (categorias nunca vistas -> ceros).
model = Pipeline(steps=[
    ("pre", pre),
    # n_estimators=300: 300 arboles de decision (mas arboles = menos varianza).
    # random_state=42: semilla para reproducibilidad.
    # n_jobs=-1: usa todos los nucleos disponibles para paralelizar el entrenamiento.
    ("reg", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])

# fit() con y_log: el modelo aprende a predecir log1p(SalePrice), no SalePrice.
# Esto es equivalente a minimizar el RMSLE (metrica oficial de Kaggle).
model.fit(X_train, ylog_train)
print("Modelo de regresion entrenado. Features:", numeric + categorical)

## Paso 3 — evaluar

Medimos las metricas tipicas de regresion sobre el precio en su escala original
(en dolares): **RMSE** (raiz del error cuadratico medio), **MAE** (error absoluto
medio) y **R2** (fraccion de varianza explicada). Ademas reportamos el **RMSE sobre
`log(SalePrice)`**, que es la **metrica oficial de la competencia de Kaggle** (y la
que el modelo optimiza directamente). Cerramos con un grafico de **predicho vs
real**.

### Interpretacion de cada metrica

| Metrica | Formula | Interpretacion |
|---|---|---|
| RMSE | $\sqrt{\frac{1}{n}\sum(y_i-\hat{y}_i)^2}$ | Error tipico en dolares; penaliza errores grandes |
| MAE | $\frac{1}{n}\sum|y_i-\hat{y}_i|$ | Error mediano en dolares; robusto a outliers |
| R2 | $1 - \frac{SS_{res}}{SS_{tot}}$ | 1.0 = perfecto; 0.0 = predice la media; puede ser negativo |
| RMSLE | RMSE sobre log1p | Error relativo; 1% de error pesa igual en casa barata o cara |

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# predict() devuelve log1p(SalePrice) porque el modelo fue entrenado en esa escala.
pred_log = model.predict(X_test)

# expm1 es la inversa de log1p: convierte de log-escala a dolares.
# expm1(x) = e^x - 1  (equivalente a np.exp(x) - 1 pero mas precisa para x cercano a 0).
pred = np.expm1(pred_log)

# Metricas sobre la escala ORIGINAL (dolares).
rmse = np.sqrt(mean_squared_error(y_test, pred))      # error cuadratico -> penaliza outliers
mae  = mean_absolute_error(y_test, pred)               # error absoluto -> mas robusto
r2   = r2_score(y_test, pred)                          # fraccion de varianza explicada

# RMSE sobre log(SalePrice): la metrica que el modelo REALMENTE optimiza.
# Es equivalente a evaluar RMSLE sobre los precios originales.
rmse_log = np.sqrt(mean_squared_error(ylog_test, pred_log))

print(f"RMSE  (USD)          : {rmse:,.0f}")
print(f"MAE   (USD)          : {mae:,.0f}")
print(f"R2                   : {r2:.4f}")
print(f"RMSE sobre log(price): {rmse_log:.4f}   <- metrica oficial de Kaggle")

In [ ]:
import matplotlib.pyplot as plt

# Grafico predicho vs real: si el modelo fuera perfecto, todos los puntos
# estarian sobre la diagonal roja (prediccion = real).
# Los puntos por encima de la diagonal = sobreestimacion.
# Los puntos por debajo = subestimacion.
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, pred, s=14, alpha=0.5, c="#4c78a8")

# Dibujamos la diagonal de prediccion perfecta.
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "--", c="#e45756", lw=1.5, label="prediccion perfecta")

ax.set_xlabel("SalePrice real")
ax.set_ylabel("SalePrice predicho")
ax.set_title(f"Predicho vs real (R2={r2:.3f})")
ax.legend()
plt.tight_layout()
plt.show()

## Paso 4 — persistir el modelo

Antes de servir predicciones, **persistimos el modelo entrenado** en disco para
reutilizarlo (lo cargaremos tal cual en el paso de serving, como haria un servicio
de inferencia).

### ¿Por que joblib?

`joblib.dump` serializa cualquier objeto Python a disco de forma eficiente,
especialmente para arrays de numpy (que son la base de los modelos de sklearn).
Es el formato estandar para persistir modelos de sklearn en produccion.

Alternativas comunes: `pickle` (mas general pero menos eficiente para arrays),
`mlflow` (para experimentos), ONNX (para interoperabilidad entre frameworks).

In [ ]:
import joblib

# Guardamos el Pipeline completo (preprocesador + modelo) en un archivo .joblib.
# Al guardar el Pipeline, se persiste TAMBIEN el OneHotEncoder con sus categorias
# aprendidas. Esto garantiza que el serving use el mismo encoding que el entrenamiento.
MODEL_PATH = os.path.join(REPO, "data", "housing_model.joblib")
joblib.dump(model, MODEL_PATH)

print(f"Modelo guardado en: {MODEL_PATH}")
print(f"Metricas -> RMSE={rmse:,.0f}  MAE={mae:,.0f}  R2={r2:.4f}  RMSE_log={rmse_log:.4f}")

## Paso 5 — serving online: inferencia con features de Feast

Entrenamos con features **historicas** (`get_historical_features`). En **produccion**,
cuando llega una peticion —"¿cuanto vale *esta* casa?"— no recalculamos todo el
historico: le pedimos al **online store** (Redis) el **ultimo** valor de cada feature
para esa entidad con **`get_online_features`** y se lo pasamos al modelo.

Este es el cierre real del ciclo: **las mismas definiciones de features** que
armaron el set de entrenamiento ahora **sirven la inferencia**, con latencia de
milisegundos y sin sesgo entrenamiento-serving.

```
   peticion (house_id) -> get_online_features (Redis) -> modelo.predict -> SalePrice
```

> Requiere haber corrido `feast apply` + `feast materialize` (Notebook 3) y tener
> Redis arriba. Si el online store no esta disponible, caemos a tomar la fila local
> para que el ejemplo siga siendo ejecutable.

### ¿Que garantiza el Pipeline en serving?

Cuando cargamos el modelo con `joblib.load` y llamamos `predict`, el Pipeline
aplica automaticamente el **mismo OneHotEncoder** que se ajusto en entrenamiento.
Si en produccion llega un barrio nuevo (no visto en train), `handle_unknown="ignore"`
lo codifica como un vector de ceros, en lugar de lanzar un error. Esto es
consistencia entrenamiento-serving garantizada por el codigo.

In [ ]:
# Cargamos el modelo persistido tal como lo haria un servicio de inferencia.
# joblib.load deserializa el Pipeline completo (preprocesador + RandomForest).
served_model = joblib.load(MODEL_PATH)

# Elegimos una casa cualquiera para simular una peticion de serving.
# En produccion, este house_id vendria en el body de la peticion HTTP.
example_id = int(train_df["house_id"].iloc[0])

def features_from_online_store(house_id):
    """Lee las ULTIMAS features de la casa desde el online store de Feast (Redis)."""
    from feast import FeatureStore
    # Instanciamos el store: Lee el registry para conocer las definiciones de features.
    store = FeatureStore(repo_path=REPO)

    # Construimos la lista de feature references (igual que en entrenamiento).
    refs = [f"house_features:{c}" for c in (numeric + categorical)]

    # get_online_features: Lee desde Redis el ULTIMO valor de cada feature
    # para el house_id dado. Latencia tipica: 1-10ms.
    online = store.get_online_features(
        features=refs,
        entity_rows=[{"house_id": house_id}],
    ).to_dict()

    # Construimos un DataFrame con las mismas columnas que X_train.
    # to_dict() devuelve los nombres sin el prefijo 'house_features:'.
    return pd.DataFrame({c: online[c] for c in (numeric + categorical)})

try:
    X_one = features_from_online_store(example_id)
    fuente = "online store de Feast (Redis)"
except Exception as e:
    # Fallback: si Feast/Redis no estan disponibles, usamos una fila del DataFrame
    # de entrenamiento. En produccion, esto nunca deberia ocurrir.
    print(f"[serving] online store no disponible ({type(e).__name__}); uso una fila local.")
    X_one = train_df.loc[train_df["house_id"] == example_id, numeric + categorical]
    fuente = "fila local (fallback)"

# El Pipeline aplica el MISMO OneHotEncoder del entrenamiento antes de predict.
# predict() devuelve log1p(SalePrice); expm1() convierte a dolares.
pred_price = float(np.expm1(served_model.predict(X_one)[0]))
print(f"Casa house_id={example_id} | features desde: {fuente}")
print(f"Prediccion de SalePrice: ${pred_price:,.0f}")

## Repaso — el ciclo cerrado

Acabamos de recorrer el ciclo completo de un sistema de ML del mundo real:

1. **Feature store (Feast)** define y sirve features de forma consistente.
2. `get_historical_features` arma un **set de entrenamiento** point-in-time
   correcto (sin leakage).
3. Un **modelo de regresion** se entrena con ese set, dentro de un pipeline que
   garantiza que el preprocesamiento sera identico en serving.
4. El modelo se **evalua** (RMSE, MAE, R2 y el RMSE sobre log(price) de Kaggle) y
   se **persiste** en disco.
5. En **serving online**, `get_online_features` lee las ultimas features desde
   Redis y el modelo predice en tiempo real — cerrando el ciclo train → serve.

**Por que importa la consistencia entrenamiento-serving:** el modelo aprende sobre
una representacion concreta de los datos. Si en produccion las features se
calculan de otra forma (otra mediana de imputacion, otro escalado, otra version de
la transformacion), el modelo recibe entradas "fuera de distribucion" y se degrada
en silencio. El feature store + el pipeline de sklearn son las dos piezas que
garantizan que *lo que se entrena es lo que se sirve*.

**Orquestacion.** Este mismo ciclo lo automatiza el DAG de **Apache Airflow** de la
plataforma compartida (`platform/dags/feature_engineering_dag.py`):
`extract -> prepare -> transform -> validate -> feast_apply -> feast_materialize ->
train_model`. Airflow es el **orquestador compartido del curso**, introducido en
este modulo.